# Experimento da aula de decisão de crédito

Base sintética, nenhum dado real de cliente. O notebook executa `experimento.py`,
confere os números que a apresentação exibe e grava `saida/resultados.json`,
`saida/metadados.json` e `saida/base_sintetica.csv.gz`.

O protocolo é temporal: treino em 2018 e 2019, validação no primeiro semestre de
2021, calibração e política no segundo semestre de 2022 e teste entre fevereiro e
julho de 2024. Cada partição só entra depois que o alvo de 12 meses maturou.

In [1]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import experimento as exp

resultados, metadados = exp.executar(verbose=False)
print("partições:", [p["nome"] for p in resultados["protocolo"]["particoes"]])
print("modelos:", sorted(resultados["avaliacao"]))

partições: ['treino', 'validacao', 'calibracao', 'teste']
modelos: ['arvore', 'boosting', 'logit', 'logit_flex']


## 1. Execução completa

`executar()` gera a base, ajusta os três modelos, calibra por Platt na partição de
calibração e avalia uma única vez no teste. Nenhum hiperparâmetro é escolhido
olhando o teste.

In [2]:
P = resultados["protocolo"]
print(f'semente {P["semente"]}')
print(f'{"partição":12s} {"início":>10s} {"fim":>10s} {"n":>6s} {"eventos":>8s} '
      f'{"taxa":>7s} {"alvo completo":>14s}')
for p in P["particoes"]:
    print(f'{p["nome"]:12s} {p["inicio"]:>10s} {p["fim"]:>10s} {p["n"]:6d} '
          f'{p["eventos"]:8d} {p["taxa"]*100:6.2f}% {p["alvo_conhecido"]:>14s}')
print("\nmedianas de imputação, calculadas só no treino:")
for k, v in P["medianas_imputacao"].items():
    print(f'  {k:8s} {v:,.4f}')

semente 20260920
partição         início        fim      n  eventos    taxa  alvo completo
treino       2018-01-01 2019-12-31  16000     2124  13.28%     2020-12-30
validacao    2021-01-01 2021-06-30   4000      522  13.05%     2022-06-30
calibracao   2022-08-01 2022-12-31   3000      464  15.47%     2023-12-31
teste        2024-02-01 2024-07-31   5000      712  14.24%     2025-07-31

medianas de imputação, calculadas só no treino:
  renda    4,569.3545
  util     43.4683


## 2. Protocolo e partições

In [3]:
M = resultados["modelos"]

# Linha da grade que corresponde ao C selecionado na validação.
def escolhido(modelo):
    return min(modelo["grade"], key=lambda g: abs(g["C"] - modelo["C"]))

for nome in ["logit", "logit_flex"]:
    g = escolhido(M[nome])
    print(f'{nome:11s} C={M[nome]["C"]:<8g} perda de validação {g["perda_validacao"]:.5f} '
          f'AUC de validação {g["auc_validacao"]:.6f}')
a = M["arvore"]
print(f'árvore        profundidade {a["profundidade"]}, mínimo por folha '
      f'{a["min_folha"]}, {a["folhas"]} folhas')
b = M["boosting"]
print(f'boosting      taxa {b["taxa"]}, profundidade {b["profundidade"]}, '
      f'{b["n_estimators"]} árvores treinadas')
print(f'              melhor iteração na validação: {b["melhor_iteracao"]}, '
      f'usada no modelo final')
print(f'              a regra de paciência {b["paciencia"]} com tolerância '
      f'{b["tolerancia"]:g} encerraria em {b["iteracao_parada"]}')
print("\ncoeficientes padronizados do logit:")
for nome, coef in zip(M["logit"]["colunas"], M["logit"]["coeficientes_padronizados"]):
    print(f'  {nome:16s} {coef:+.4f}')

logit       C=0.01     perda de validação 0.34299 AUC de validação 0.737944
logit_flex  C=100      perda de validação 0.34395 AUC de validação 0.738463
árvore        profundidade 6, mínimo por folha 300, 24 folhas
boosting      taxa 0.1, profundidade 2, 300 árvores treinadas
              melhor iteração na validação: 106, usada no modelo final
              a regra de paciência 20 com tolerância 0.0001 encerraria em 89

coeficientes padronizados do logit:
  renda            -0.1047
  comp             +0.4772
  rel              -0.1687
  util             +0.1886
  hist             +0.4019
  canal_digital    +0.0548
  canal_parceiro   +0.1125
  renda_ausente    +0.0002
  util_ausente     +0.0117


## 3. Seleção de cada modelo, decidida na validação

In [4]:
A = resultados["avaliacao"]
print(f'{"modelo":12s} {"AUC":>9s} {"KS":>7s} {"Brier":>9s} {"log loss":>9s} '
      f'{"n":>6s} {"eventos":>8s}')
for k in ["logit", "logit_flex", "arvore", "boosting"]:
    t = A[k]["teste"]
    print(f'{k:12s} {t["auc"]:9.6f} {t["ks"]:7.4f} {t["brier_calibrada"]:9.5f} '
          f'{t["log_loss_calibrada"]:9.5f} {t["n"]:6d} {t["eventos"]:8d}')
aucs = [A[k]["teste"]["auc"] for k in ["logit", "arvore", "boosting"]]
print(f'\namplitude entre os três modelos principais: {max(aucs)-min(aucs):.6f}')

modelo             AUC      KS     Brier  log loss      n  eventos
logit         0.744562  0.3758   0.10670   0.35711   5000      712
logit_flex    0.745322  0.3781   0.10649   0.35644   5000      712
arvore        0.739170  0.3625   0.10823   0.36068   5000      712
boosting      0.744367  0.3769   0.10707   0.35790   5000      712

amplitude entre os três modelos principais: 0.005392


## 4. Resultado no teste

Os três modelos ficam próximos. A aula não tem vencedor predeterminado: a
diferença entre o maior e o menor AUC é menor que a variabilidade de uma amostra
única deste tamanho.

In [5]:
for k in ["logit", "arvore", "boosting"]:
    t = A[k]["teste"]
    print(f'{k:10s} AUC bruta {t["auc"]:.6f}  AUC calibrada {t["auc_calibrada"]:.6f}  '
          f'iguais: {t["auc"] == t["auc_calibrada"]}')
print("\nfaixas de calibração do boosting no teste, previsto contra observado:")
for i, f in enumerate(A["boosting"]["calibracao_teste_calibrada"], 1):
    print(f'  faixa {i:2d}  n={f["n"]:4d}  previsto {f["prev_media"]*100:6.2f}%  '
          f'observado {f["obs"]*100:6.2f}%  Wilson '
          f'[{f["wilson"][0]*100:5.2f}%, {f["wilson"][1]*100:5.2f}%]')

logit      AUC bruta 0.744562  AUC calibrada 0.744562  iguais: True
arvore     AUC bruta 0.739170  AUC calibrada 0.739170  iguais: True
boosting   AUC bruta 0.744367  AUC calibrada 0.744367  iguais: True

faixas de calibração do boosting no teste, previsto contra observado:
  faixa  1  n= 500  previsto   4.07%  observado   2.60%  Wilson [ 1.53%,  4.40%]
  faixa  2  n= 501  previsto   5.95%  observado   4.99%  Wilson [ 3.40%,  7.26%]
  faixa  3  n= 504  previsto   7.40%  observado   6.15%  Wilson [ 4.37%,  8.60%]
  faixa  4  n= 495  previsto   8.83%  observado  10.10%  Wilson [ 7.75%, 13.07%]
  faixa  5  n= 500  previsto  10.47%  observado   8.40%  Wilson [ 6.27%, 11.16%]
  faixa  6  n= 500  previsto  12.38%  observado  10.00%  Wilson [ 7.67%, 12.94%]
  faixa  7  n= 503  previsto  14.59%  observado  11.73%  Wilson [ 9.20%, 14.84%]
  faixa  8  n= 497  previsto  17.99%  observado  21.13%  Wilson [17.77%, 24.93%]
  faixa  9  n= 500  previsto  25.29%  observado  22.80%  Wilson [19.34%, 26.6

## 5. Calibração

A calibração de Platt é ajustada na partição de calibração e aplicada ao teste sem
reajuste. Por ser monótona, ela não altera a ordenação nem a AUC: a igualdade
abaixo é exata.

In [6]:
E = resultados["explicacao"]
print(f'referência do fundo: F = {E["locais"][0]["base"]:.6f}')
for loc in E["locais"]:
    print(f'{loc["cliente"]:6s} escore {loc["escore"]:+.6f}  '
          f'base + contribuições {loc["conferencia_soma"]:+.6f}  '
          f'erro {loc["erro_soma"]:.2e}  PD calibrada {loc["pd_calibrada"]*100:.2f}%')
print("\ncontribuições de Bruno, em escore:")
for g, v in sorted(E["locais"][1]["contribuicoes"].items(), key=lambda x: -abs(x[1])):
    print(f'  {g:8s} {v:+.6f}')

referência do fundo: F = -2.055770
Ana    escore -2.945797  base + contribuições -2.945797  erro 0.00e+00  PD calibrada 5.49%
Bruno  escore -1.189179  base + contribuições -1.189179  erro 0.00e+00  PD calibrada 25.78%
Carla  escore -1.539569  base + contribuições -1.539569  erro 0.00e+00  PD calibrada 19.56%
Diego  escore +0.453838  base + contribuições +0.453838  erro 0.00e+00  PD calibrada 64.91%

contribuições de Bruno, em escore:
  hist     +0.650562
  util     +0.155027
  rel      +0.120902
  canal    -0.031184
  comp     -0.024405
  renda    -0.004310


## 6. Explicação local: Shapley interventional exato

Seis grupos semânticos de características, 64 coalizões, valor exato. A propriedade
de eficiência exige que a base somada às contribuições reproduza o escore do
cliente.

In [7]:
pol = resultados["politica"]
corte = pol["logit"]["corte_congelado"]
eco = resultados["protocolo"]["economia"]
# Formatação em português: milhar com ponto e decimal com vírgula.
def reais(v, casas=2):
    txt = f'{abs(v):,.{casas}f}'.replace(",", "@").replace(".", ",").replace("@", ".")
    return ("−R$ " if v < 0 else "R$ ") + txt

print(f'margem {reais(eco["margem"], 0)} · LGD {eco["lgd"]*100:.0f}% · '
      f'EAD {reais(eco["ead"], 0)} · equilíbrio {eco["corte_equilibrio"]*100:.1f}%')
print(f'\ncorte congelado {corte*100:.0f}%')
print(f'{"modelo":10s} {"aprovação":>10s} {"inadimplência":>14s} {"por aprovado":>14s} '
      f'{"total":>14s}')
for k in ["logit", "arvore", "boosting"]:
    linha = min(pol[k]["teste"], key=lambda l: abs(l["corte"] - corte))
    print(f'{k:10s} {linha["aprovacao"]*100:9.2f}% {linha["inadimplencia"]*100:13.2f}% '
          f'{reais(linha["resultado_medio"]):>16s} '
          f'{reais(linha["resultado_total"], 0):>18s}')

margem R$ 1.200 · LGD 60% · EAD R$ 10.000 · equilíbrio 20.0%

corte congelado 20%
modelo      aprovação  inadimplência   por aprovado          total
logit          74.78%          8.50%        R$ 689,70       R$ 2.578.800
arvore         77.66%          9.32%        R$ 640,64       R$ 2.487.600
boosting       79.36%          9.22%        R$ 646,57       R$ 2.565.600


## 7. Política congelada

Margem de R$ 1.200, LGD de 60% e EAD de R$ 10.000 dão ponto de equilíbrio em PD de
20%. O corte é congelado antes do teste e aplicado igualmente aos três modelos.

In [8]:
import hashlib

for k, v in metadados.items():
    print(f'{k:20s} {v}')

# Conferência independente: os resumos gravados batem com os arquivos em disco.
base = Path("saida")
for arquivo, chave in [("resultados.json", "sha256_resultados"),
                       ("base_sintetica.csv.gz", "sha256_base")]:
    atual = hashlib.sha256((base / arquivo).read_bytes()).hexdigest()
    print(f'\n{arquivo}: {"confere" if atual == metadados[chave] else "DIVERGE"}')
    print(f'  gravado {metadados[chave]}')
    print(f'  disco   {atual}')

gerado_por           experimento.py
semente              20260920
python               3.11.15
plataforma           Linux-6.18.44-fc-v37-x86_64-with-glibc2.39
numpy                2.4.6
scikit_learn         1.9.1
arquivo_resultados   saida/resultados.json
sha256_resultados    819746e8a01e72c2e9974080324900247cbae0f2497482e62c3184a5a6389b2e
bytes_resultados     388909
arquivo_base         saida/base_sintetica.csv.gz
sha256_base          fdf082bbb42ca53d038a875a634f98d7aab4536407c2ce8639322d595e96c75f
linhas_base          28000
observacao           Base sintética. Nenhum dado real de cliente. Os modelos são ajustados no treino, selecionados na validação, calibrados na partição de calibração e avaliados uma única vez no teste.

resultados.json: confere
  gravado 819746e8a01e72c2e9974080324900247cbae0f2497482e62c3184a5a6389b2e
  disco   819746e8a01e72c2e9974080324900247cbae0f2497482e62c3184a5a6389b2e

base_sintetica.csv.gz: confere
  gravado fdf082bbb42ca53d038a875a634f98d7aab4536407c2ce86

## 8. Reprodutibilidade

Semente, versões e resumo criptográfico dos arquivos gerados.